In [31]:
import os

BASE_DIR = os.getcwd()

print(BASE_DIR)

C:\Users\Analysis_VM\Desktop\Ransomware_Project\notebooks


In [1]:
# ================================
# Mapping Files Paths
# ================================

import os
import json

MASTER_MAPPING_FILE = "family_mapping.json"

CLEAN_MAPPING_FILE = "family_mapping_clean.json"

HASH_FILE = "hashes.txt"

DATASET_FOLDER = "../dataset/ransomware"

In [2]:
# ================================
# Check Missing Hashes
# ================================

with open(MASTER_MAPPING_FILE, "r") as f:
    master_mapping = json.load(f)

folder_hashes = set()

for f_name in os.listdir(DATASET_FOLDER):

    if f_name.endswith((".exe", ".dll")):

        sha256 = f_name.rsplit(".", 1)[0]

        folder_hashes.add(sha256)

mapping_hashes = set(master_mapping.keys())

missing_in_mapping = folder_hashes - mapping_hashes

missing_in_folder = mapping_hashes - folder_hashes

print(f"[INFO] Files in folder: {len(folder_hashes)}")
print(f"[INFO] Hashes in mapping: {len(mapping_hashes)}")

print(f"\n[INFO] Missing in mapping: {len(missing_in_mapping)}")
print(f"[INFO] Missing in folder: {len(missing_in_folder)}")

print("\n[INFO] Sample missing hashes:")
print(list(missing_in_mapping)[:10])

# ================================
# Remove orphan hashes
# ================================

extra_hashes = mapping_hashes - folder_hashes

if extra_hashes:

    print(f"\n[INFO] Removing {len(extra_hashes)} orphan hashes...")

    for sha in extra_hashes:

        del master_mapping[sha]

    with open(MASTER_MAPPING_FILE, "w") as f:

        json.dump(master_mapping, f, indent=4)

    print("[SUCCESS] Mapping cleaned")

else:

    print("\n[INFO] No orphan hashes found")

[INFO] Files in folder: 1500
[INFO] Hashes in mapping: 1504

[INFO] Missing in mapping: 0
[INFO] Missing in folder: 4

[INFO] Sample missing hashes:
[]

[INFO] Removing 4 orphan hashes...
[SUCCESS] Mapping cleaned


In [3]:
# ================================
# Sync Clean Mapping With Master
# ================================
import json

with open(MASTER_MAPPING_FILE, "r") as f:
    master_mapping = json.load(f)

with open(CLEAN_MAPPING_FILE, "w") as f:
    json.dump(master_mapping, f, indent=4)

print("[SUCCESS] clean mapping synced with master")
print(f"[INFO] Total hashes: {len(master_mapping)}")

[SUCCESS] clean mapping synced with master
[INFO] Total hashes: 1500


In [4]:
# ================================
# Merge New Hashes Into Clean Mapping
# Run it After any new downloaded file
# ================================

import json

with open(MASTER_MAPPING_FILE, "r") as f:
    master_mapping = json.load(f)

with open(CLEAN_MAPPING_FILE, "r") as f:
    clean_mapping = json.load(f)

added = 0

for sha, family in master_mapping.items():

    if sha not in clean_mapping:

        clean_mapping[sha] = family
        added += 1

with open(CLEAN_MAPPING_FILE, "w") as f:
    json.dump(clean_mapping, f, indent=4)

print(f"[INFO] Added {added} new hashes")
print(f"[INFO] Total hashes now: {len(clean_mapping)}")

[INFO] Added 0 new hashes
[INFO] Total hashes now: 1500


In [5]:
# ================================
# Clean Invalid Mapping Entries
# Run it if there are unknown / none labels
# ================================

import json

with open(CLEAN_MAPPING_FILE, "r") as f:
    clean_mapping = json.load(f)

before_count = len(clean_mapping)

clean_mapping = {

    k: v.strip().lower()

    for k, v in clean_mapping.items()

    if (
        v
        and isinstance(v, str)
        and str(v).lower() not in ["none", "unknown", ""]
    )
}

after_count = len(clean_mapping)

removed = before_count - after_count

print(f"[INFO] Before clean: {before_count}")
print(f"[INFO] After clean: {after_count}")
print(f"[INFO] Removed invalid entries: {removed}")

with open(CLEAN_MAPPING_FILE, "w") as f:
    json.dump(clean_mapping, f, indent=4)

print("[SUCCESS] Clean mapping saved")

[INFO] Before clean: 1500
[INFO] After clean: 1500
[INFO] Removed invalid entries: 0
[SUCCESS] Clean mapping saved


In [6]:
# ================================
# Load Clean Mapping
# ================================

import os
import json
import pandas as pd

DATASET_FOLDER = "../dataset/ransomware"

with open(CLEAN_MAPPING_FILE, "r") as f:
    clean_mapping = json.load(f)

print("[INFO] Loaded mapping:", len(clean_mapping))

[INFO] Loaded mapping: 1500


In [7]:
import os

for label in ["benign", "ransomware"]:

    folder_path = os.path.join(
        "../dataset",
        label
    )

    if not os.path.exists(folder_path):

        print(f"[CRITICAL] Missing folder: {folder_path}")

        continue

    count = 0

    for root, dirs, files in os.walk(folder_path):

        for file_name in files:

            if file_name.lower().endswith(
                (".exe", ".dll")
            ):

                count += 1

    print(f"[INFO] Raw {label} PE files: {count}")

[INFO] Raw benign PE files: 936
[INFO] Raw ransomware PE files: 1500


In [8]:
# ================================
# CELL 2: Build Clean File List
# ================================

import os
import hashlib
import json

DATASET_FOLDER = "../dataset/ransomware"

with open(CLEAN_MAPPING_FILE, "r") as f:
    clean_mapping = json.load(f)

def calculate_sha256(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:

        while chunk := f.read(8192):
            sha256.update(chunk)

    return sha256.hexdigest()

clean_files = []

for file_name in sorted(os.listdir(DATASET_FOLDER)):

    if not file_name.endswith((".exe", ".dll")):
        continue

    file_path = os.path.join(DATASET_FOLDER, file_name)

    try:

        sha256 = calculate_sha256(file_path)

        family = clean_mapping.get(sha256)

        if not family:
            continue

        clean_files.append((file_name, family))

    except Exception as e:

        print(f"[ERROR] {file_name}: {e}")

print("[INFO] Clean files:", len(clean_files))

[INFO] Clean files: 1500


In [9]:
# ================================
# CELL 3: Analyze Distribution
# ================================

from collections import Counter

families = [fam for _, fam in clean_files]

counter = Counter(families)

print("\n[INFO] Top Families:\n")

for fam, count in counter.most_common(15):

    print(f"{fam:<20} -> {count}")


[INFO] Top Families:

blackmatter          -> 195
wannacry             -> 161
raworld              -> 145
akira                -> 145
trigona              -> 114
lockbit              -> 108
medusa               -> 75
qilin                -> 59
cactus               -> 50
blackcat             -> 49
hellokitty           -> 48
bianlian             -> 37
rhysida              -> 27
dragonforce          -> 20
fog                  -> 17


In [10]:
# ================================
# CELL 4: Filter + Balance Dataset
# ================================
from collections import defaultdict
import random

random.seed(42)

TOP_N = None
MAX_PER_FAMILY = 80
MIN_PER_FAMILY = 10

# Families مش ransomware — لازم تتشال
EXCLUDE_FAMILIES = {
    "rustystealer",
    "neshta",
    "nitrogen",
}

TOP_FAMILIES = [
    fam
    for fam, _ in counter.most_common(TOP_N)
]

filtered_files = [
    (f, fam)
    for (f, fam) in clean_files
    if fam in TOP_FAMILIES
    and fam not in EXCLUDE_FAMILIES
]

grouped = defaultdict(list)

for f, fam in filtered_files:
    grouped[fam].append((f, fam))

balanced_files = []
selected_counts = {}

for fam, items in grouped.items():

    if len(items) < MIN_PER_FAMILY:
        continue

    selected = random.sample(
        items,
        min(len(items), MAX_PER_FAMILY)
    )

    balanced_files.extend(selected)
    selected_counts[fam] = len(selected)

print("[INFO] Balanced files:", len(balanced_files))
print("\n[INFO] Distribution after balancing:\n")

for fam, count in sorted(
    selected_counts.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{fam:<20} -> {count}")

[INFO] Balanced files: 890

[INFO] Distribution after balancing:

raworld              -> 80
trigona              -> 80
wannacry             -> 80
akira                -> 80
lockbit              -> 80
blackmatter          -> 80
medusa               -> 75
qilin                -> 59
cactus               -> 50
blackcat             -> 49
hellokitty           -> 48
bianlian             -> 37
rhysida              -> 27
dragonforce          -> 20
fog                  -> 17
inc                  -> 14
chaos                -> 14


In [11]:
# =====================================
# CELL 5: Feature Extraction (v2)
# New features: has_version_info, has_debug, wx_sections
# =====================================

import os
import pefile
import pandas as pd
import math
import hashlib
import json
import shutil

from collections import Counter

# ================================
# Output File
# ================================
OUTPUT_FILE = "../data/pe_features(All_families_v2).csv"

# ================================
# Load Family Mapping
# ================================
with open(CLEAN_MAPPING_FILE, "r") as f:
    clean_mapping = json.load(f)

# ================================
# Balanced ransomware set
# NOTE: uses real SHA256 from file contents
# ================================
def calculate_sha256(file_path):
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(8192):
            sha256.update(chunk)
    return sha256.hexdigest()

RANSOMWARE_FOLDER = "../dataset/ransomware"

balanced_fnames = set(fname for fname, _ in balanced_files)

balanced_set = set()
for fname in balanced_fnames:
    fpath = os.path.join(RANSOMWARE_FOLDER, fname)
    if os.path.exists(fpath):
        real_sha256 = calculate_sha256(fpath)
        balanced_set.add(real_sha256)

print(f"[INFO] balanced_set size: {len(balanced_set)}")

# ================================
# Suspicious APIs
# ================================
CRYPTO_APIS = {
    "CryptEncrypt",
    "CryptDecrypt",
    "CryptGenKey",
    "CryptAcquireContext",
    "BCryptEncrypt",
    "BCryptDecrypt",
    "BCryptGenerateSymmetricKey",
    "BCryptOpenAlgorithmProvider",
}

FILE_OPS = {
    "MoveFileExW",
    "DeleteFileW",
}

# ================================
# Dataset Container
# ================================
data = []

# ================================
# Entropy Function
# ================================
def calculate_entropy(byte_data):
    if not byte_data:
        return 0
    byte_counts = Counter(byte_data)
    length = len(byte_data)
    entropy = 0
    for count in byte_counts.values():
        p_x = count / length
        entropy -= p_x * math.log2(p_x)
    return entropy

# ================================
# Validate Dataset Folders
# ================================
for label in ["benign", "ransomware"]:
    folder = os.path.join("../dataset", label)
    if not os.path.exists(folder):
        raise FileNotFoundError(f"[CRITICAL] Missing folder: {folder}")
    pe_count = 0
    for root, dirs, files in os.walk(folder):
        for file_name in files:
            if file_name.lower().endswith((".exe", ".dll")):
                pe_count += 1
    print(f"[INFO] {label}: {pe_count} PE files found")

# ================================
# Loop Through Dataset
# ================================
for label in ["benign", "ransomware"]:

    folder_path = os.path.join("../dataset", label)

    for root, dirs, files in os.walk(folder_path):

        for file_name in sorted(files):

            if not file_name.lower().endswith((".exe", ".dll")):
                continue

            file_path = os.path.join(root, file_name)

            pe = None

            try:

                # SHA256
                sha256 = calculate_sha256(file_path)

                # Family
                if label == "ransomware":
                    if sha256 not in balanced_set:
                        continue
                    family = clean_mapping.get(sha256, None)
                    if family is None:
                        print(f"[WARNING] Missing family: {file_name}")
                        continue
                else:
                    family = "benign"
                    if sha256 in clean_mapping:
                        print(f"[WARNING] Benign in ransomware mapping: {file_name}")
                        continue

                # Load PE
                pe = pefile.PE(file_path, fast_load=True)

                pe.parse_data_directories(
                    directories=[
                        pefile.DIRECTORY_ENTRY['IMAGE_DIRECTORY_ENTRY_IMPORT'],
                        pefile.DIRECTORY_ENTRY['IMAGE_DIRECTORY_ENTRY_RESOURCE'],
                    ]
                )

                # Basic Info
                file_size     = os.path.getsize(file_path)
                num_sections  = len(pe.sections)
                many_sections = 1 if num_sections > 6 else 0
                size_of_image = pe.OPTIONAL_HEADER.SizeOfImage
                size_of_code  = pe.OPTIONAL_HEADER.SizeOfCode
                entry_point   = pe.OPTIONAL_HEADER.AddressOfEntryPoint

                # Imports Analysis
                num_imports  = 0
                num_dlls     = 0
                has_crypto   = 0
                has_bcrypt   = 0
                has_file_ops = 0

                if hasattr(pe, 'DIRECTORY_ENTRY_IMPORT'):
                    num_dlls = len(pe.DIRECTORY_ENTRY_IMPORT)
                    for entry in pe.DIRECTORY_ENTRY_IMPORT:
                        num_imports += len(entry.imports)
                        for imp in entry.imports:
                            if imp.name:
                                name = imp.name.decode(errors="ignore")
                                if name in CRYPTO_APIS:
                                    has_crypto = 1
                                if name.startswith("BCrypt"):
                                    has_bcrypt = 1
                                if name in FILE_OPS:
                                    has_file_ops = 1

                # Entropy
                section_entropies = []
                for section in pe.sections:
                    section_data = section.get_data()
                    if not section_data:
                        section_entropies.append(0.0)
                        continue
                    section_entropies.append(calculate_entropy(section_data))

                avg_entropy = sum(section_entropies) / len(section_entropies) if section_entropies else 0
                max_entropy = max(section_entropies) if section_entropies else 0
                min_entropy = min(section_entropies) if section_entropies else 0

                # ================================
                # New Features
                # ================================

                # has_version_info — RT_VERSION = resource type 16
                has_version_info = 0
                if hasattr(pe, 'DIRECTORY_ENTRY_RESOURCE'):
                    for res_type in pe.DIRECTORY_ENTRY_RESOURCE.entries:
                        if hasattr(res_type, 'id') and res_type.id == 16:
                            has_version_info = 1
                            break

                # has_debug — DATA_DIRECTORY index 6
                try:
                    has_debug = int(pe.OPTIONAL_HEADER.DATA_DIRECTORY[6].Size > 0)
                except Exception:
                    has_debug = 0

                # wx_sections — writeable + executable
                wx_sections = sum(
                    1 for s in pe.sections
                    if (s.Characteristics & 0x20000000) and
                       (s.Characteristics & 0x80000000)
                )

                # Append
                data.append([
                    sha256,
                    file_name,
                    family,
                    file_size,
                    num_sections,
                    many_sections,
                    size_of_image,
                    size_of_code,
                    entry_point,
                    num_imports,
                    num_dlls,
                    avg_entropy,
                    max_entropy,
                    min_entropy,
                    has_crypto,
                    has_bcrypt,
                    has_file_ops,
                    has_version_info,
                    has_debug,
                    wx_sections,
                    0 if label == "benign" else 1
                ])

            except pefile.PEFormatError:
                print(f"[SKIP] Invalid PE: {file_name}")
                INVALID_FOLDER = "../invalid_pe"
                os.makedirs(INVALID_FOLDER, exist_ok=True)
                try:
                    shutil.move(file_path, os.path.join(INVALID_FOLDER, file_name))
                    print(f"[MOVED] {file_name} -> invalid_pe")
                except Exception as move_error:
                    print(f"[ERROR] Move failed: {move_error}")

            except OSError as e:
                print(f"[SKIP] File error: {file_name}: {e}")

            except Exception as e:
                import traceback
                print(f"[ERROR] Unexpected: {file_name}")
                traceback.print_exc()

            finally:
                if pe is not None:
                    pe.close()
                    pe = None

# ================================
# Create DataFrame
# ================================
columns = [
    "sha256",
    "file_name",
    "family",
    "file_size",
    "num_sections",
    "many_sections",
    "size_of_image",
    "size_of_code",
    "entry_point",
    "num_imports",
    "num_dlls",
    "avg_entropy",
    "max_entropy",
    "min_entropy",
    "has_crypto",
    "has_bcrypt",
    "has_file_ops",
    "has_version_info",
    "has_debug",
    "wx_sections",
    "label"
]

df = pd.DataFrame(data, columns=columns)

# ================================
# Save CSV
# ================================
df.to_csv(OUTPUT_FILE, index=False)

# ================================
# Final Logs
# ================================
print("\n[SUCCESS] Extraction completed")
print("[INFO] Extracted samples:", len(data))
print("[INFO] Output saved to:", OUTPUT_FILE)

# ================================
# ML Warning
# ================================
metadata_cols = ["sha256", "file_name", "family"]
feature_cols  = [c for c in df.columns if c not in metadata_cols + ["label"]]

print("\n[INFO] Metadata columns (DROP before ML):", metadata_cols)
print("[INFO] Feature columns:", feature_cols)
print("[INFO] Target column: label")

# ================================
# Quick Validation
# ================================
print("\n[INFO] Label distribution:")
print(df["label"].value_counts())

print("\n[INFO] Family distribution:")
print(df["family"].value_counts())

print("\n[INFO] New features averages by label:")
print(df.groupby("label")[["has_version_info","has_debug","wx_sections"]].mean().round(3))

print("\n[INFO] NaN check:")
print(df[feature_cols].isnull().sum())

[INFO] balanced_set size: 890
[INFO] benign: 936 PE files found
[INFO] ransomware: 1500 PE files found

[SUCCESS] Extraction completed
[INFO] Extracted samples: 1826
[INFO] Output saved to: ../data/pe_features(All_families_v2).csv

[INFO] Metadata columns (DROP before ML): ['sha256', 'file_name', 'family']
[INFO] Feature columns: ['file_size', 'num_sections', 'many_sections', 'size_of_image', 'size_of_code', 'entry_point', 'num_imports', 'num_dlls', 'avg_entropy', 'max_entropy', 'min_entropy', 'has_crypto', 'has_bcrypt', 'has_file_ops', 'has_version_info', 'has_debug', 'wx_sections']
[INFO] Target column: label

[INFO] Label distribution:
label
0    936
1    890
Name: count, dtype: int64

[INFO] Family distribution:
family
benign         936
raworld         80
trigona         80
wannacry        80
lockbit         80
blackmatter     80
akira           80
medusa          75
qilin           59
cactus          50
blackcat        49
hellokitty      48
bianlian        37
rhysida         27
d